In [263]:
from pathlib import Path
import logging
import math
import os
import shutil
import subprocess
import textwrap
import traceback
import zipfile
import nemosis
import pandas as pd
import io
import urllib.request
import openpyxl

logging.getLogger("nemosis").setLevel(logging.WARNING)


In [264]:
"""
A generic scheduler function to fetch datasource data by months
and skip re pulling existing months
"""

def _month_compiler(
        datasource_fetch_function: callable,
        start_date:pd.Timestamp,
        end_date:pd.Timestamp,
        datasource_file_path:Path
        ):
    
    datasource_file_year_month_range = set()
    # Check if the datasource file already exists, if so, check which years/months it has
    if datasource_file_path.exists():
        # Extract only the SETTLEMENTDATE index as a series
        datasource_file_date_range = pd.read_csv(datasource_file_path, usecols=[0]).iloc[:, 0]
        # Create a 'set' of unique year-months from datasource_file_date_range
        datasource_file_year_month_range = set(pd.to_datetime(datasource_file_date_range, format="mixed").dt.to_period("M"))
        print(f"  Found {len(datasource_file_year_month_range)} already-processed month(s) — will skip.", flush=True)

    # Set a var for total required data range
    datasource_required_total_range = pd.date_range(start_date, end_date - pd.Timedelta(days=1), freq="MS")

    # Loop through the datasource_required_total_range
    for i, month in enumerate(datasource_required_total_range):
        # Month in datasource_file_year_month_range so skip iteration
        if month.to_period("M") in datasource_file_year_month_range:
            print(f"{i + 1:3d}/{len(datasource_required_total_range)} {month:%Y-%m} skipping (already processed).", flush=True)
            continue
        # Month NOT in datasource_file_year_month_range so call datasource fetch function
        else:
            # Define the start of the calander month  
            current_month_start = month.strftime("%Y/%m/%d %H:%M:%S")
            # Define the end of the calander month
            current_month_end = (month + pd.offsets.MonthEnd(0) + pd.Timedelta(days=1)).strftime("%Y/%m/%d %H:%M:%S")
            
            print(f"{i + 1:3d}/{len(datasource_required_total_range)} {month:%Y-%m} fetching...", flush=True)
            
            # Call the datasource fetch function for the current month
            datasource_data = datasource_fetch_function(current_month_start, current_month_end)

            # Merge with existing datasource file if it exists
            if datasource_file_path.exists():
                datasource_file = pd.read_csv(datasource_file_path, index_col=0, low_memory=False)
                datasource_file.index = pd.to_datetime(datasource_file.index, format="mixed")
                datasource_data = pd.concat([datasource_file, datasource_data])
                datasource_data = datasource_data[~datasource_data.index.duplicated(keep="last")].sort_index()

            # Ensure the index name is set correctly
            datasource_data.index.name = "SETTLEMENTDATE"
            # Save after every new month
            datasource_data.to_csv(datasource_file_path)

            # Empty the temporary cache (Deletes all files and subdirectories)
            for entry in Path("Pre_processing/temporary_cache").iterdir():
                if entry.is_dir():
                    shutil.rmtree(entry)
                else:
                    entry.unlink()

            print(f"  {month:%Y-%m} saved & raw files deleted.", flush=True)

    return pd.read_csv(datasource_file_path, index_col=0, parse_dates=True, low_memory=False)


In [265]:
"""
A generic function to fetch datasource data in one shot
"""

def _one_shot_compiler(
        datasource_fetch_function: callable,
        start_date:pd.Timestamp,
        end_date:pd.Timestamp,
        datasource_file_path:Path
        ):
    
    
        datasource_data = datasource_fetch_function(start_date, end_date)
        datasource_data.index.name = "SETTLEMENTDATE"
        datasource_data.to_csv(datasource_file_path)

        return pd.read_csv(datasource_file_path, index_col=0, parse_dates=True)


In [266]:
"""
Datasource 1
 - Datasource origin: nemosis
 - Datasource name: dispatch price     : AEMO's DISPATCHPRICE table — contains the Regional Reference Price (RRP) and related pricing outcomes published for each region at the end of every 5-minute dispatch interval
 - Variables:
    RRP         : Regional Reference Price ($/MWh) — the spot price for electricity in each NEM region, set by the market clearing process for each 5-minute dispatch interval
    REGIONID    : NEM region identifier used to filter and pivot the data (e.g. NSW1, QLD1, VIC1, SA1)
    """
def _dispatch_price(start: str, end: str):
    region_columns = {"NSW1": "nsw_price", "QLD1": "qld_price", "VIC1": "vic_price", "SA1": "sa_price"}

    def _API_call():

        API_response = nemosis.dynamic_data_compiler(
            start_time=start,
            end_time=end,
            table_name="DISPATCHPRICE",
            raw_data_location="Pre_processing/temporary_cache",
            select_columns=["SETTLEMENTDATE", "REGIONID", "RRP"],
            filter_cols=["REGIONID"],
            filter_values=[list(region_columns.keys())],
            fformat="feather",
            keep_csv=False,
        )
        
        return API_response

    def _clean_up(API_response):
        # Handle conversion of data types
        API_response["SETTLEMENTDATE"] = pd.to_datetime(API_response["SETTLEMENTDATE"])
        API_response["RRP"] = pd.to_numeric(API_response["RRP"], errors="coerce")

        # Handle data granularity
        API_response = (
            API_response.groupby(["SETTLEMENTDATE", "REGIONID"])["RRP"]
            .mean()
            .unstack("REGIONID")
            .resample("5min").mean()
            .asfreq("5min")
        )

        # Rename columns
        API_response = API_response.rename(columns=region_columns)

        return API_response

    data = _API_call()
    data_clean = _clean_up(data)
    return data_clean


In [267]:
"""
Datasource 2
 - Datasource origin: nemosis
 - Datasource name: dispatch region sum    : AEMO's DISPATCHREGIONSUM table — contains region-level demand, generation, and interconnector summary statistics published at the end of every 5-minute dispatch interval
 - Variables:
    TOTALDEMAND               : Total metered demand (MW) for the region in the dispatch interval
    AVAILABLEGENERATION       : Total available generation capacity (MW) that could be dispatched in the region
    NETINTERCHANGE            : Net interconnector flow (MW) into the region — positive means imports, negative means exports
    DEMANDFORECAST            : AEMO's forecast of regional demand (MW) for the dispatch interval
    DISPATCHABLEGENERATION    : Total generation (MW) actually dispatched in the region for the interval

    REGIONID                  : NEM region identifier used to filter and pivot the data (e.g. NSW1, QLD1, VIC1, SA1)
"""
def _dispatch_region_sum(start: str, end: str):
    RAW_FIELDS    = ["TOTALDEMAND", "AVAILABLEGENERATION", "NETINTERCHANGE", "DEMANDFORECAST", "DISPATCHABLEGENERATION"]
    REGION_SUFFIX = {"NSW1": "_nsw", "QLD1": "_qld", "VIC1": "_vic", "SA1": "_sa"}

    def _API_call():
        
        API_response = nemosis.dynamic_data_compiler(
            start_time=start,
            end_time=end,
            table_name="DISPATCHREGIONSUM",
            raw_data_location="Pre_processing/temporary_cache",
            select_columns=["SETTLEMENTDATE", "REGIONID"] + RAW_FIELDS,
            filter_cols=["REGIONID"],
            filter_values=[list(REGION_SUFFIX.keys())],
            fformat="feather",
            keep_csv=False,
        )
        
        return API_response

    def _clean_up(API_response):

        # Handle conversion of data types
        OUTPUT_FIELDS = ["demand", "avail_gen", "interchange", "demand_forecast", "dispatch_gen"]

        API_response["SETTLEMENTDATE"] = pd.to_datetime(API_response)

        for col in RAW_FIELDS:
            API_response[col] = pd.to_numeric(API_response[col], errors="coerce")

        # Handle data granularity / transform from long format
        region_frames = []
        for region, suffix in REGION_SUFFIX.items():
            rdf = API_response[API_response["REGIONID"] == region][["SETTLEMENTDATE"] + RAW_FIELDS].copy()
            rdf = (
                rdf.set_index("SETTLEMENTDATE")
                .sort_index()
                .resample("5min")
                .mean(numeric_only=True)
            )
            rdf.columns = [f"{name}{suffix}" for name in OUTPUT_FIELDS]
            region_frames.append(rdf)
            
        API_response = pd.concat(region_frames, axis=1).sort_index()

        return API_response

    data = _API_call()
    data_clean = _clean_up(data)
    return data_clean


In [268]:
"""
Datasource 3
 - Datasource origin: nemosis + www.aemo.com.au
 - Datasource name: generation fuel   : Per-fuel-type aggregated generation (MW) from AEMO's DISPATCH_UNIT_SCADA table,
                                         with DUID-to-fuel mapping sourced from the NEM Registration and Exemption List.
 - Variables:
    coal_mw_{region}              : Total coal generation (MW)
    wind_mw_{region}              : Total wind generation (MW)
    solar_mw_{region}             : Total solar/PV generation (MW)
    hydro_mw_{region}             : Total hydro generation (MW)
    gas_mw_{region}               : Total gas/diesel/distillate generation (MW)
    battery_charge_mw_{region}    : Total battery charging (MW, positive)
    battery_discharge_mw_{region} : Total battery discharging (MW, positive)
"""
def _generation_fuel(start: str, end: str):
    REGION_TO_SUFFIX = {"NSW1": "nsw", "QLD1": "qld", "VIC1": "vic", "SA1": "sa"}

    def _build_mapping():
        cache_path = Path("Pre_processing/temporary_cache") / "NEM Registration and Exemption List.xlsx"
        url = "https://www.aemo.com.au/-/media/files/electricity/nem/participant_information/nem-registration-and-exemption-list.xlsx"
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp, open(cache_path, "wb") as f:
            f.write(resp.read())
        try:
            df = pd.read_excel(cache_path, sheet_name="PU and Scheduled Loads", engine="openpyxl", header=0)
        finally:
            cache_path.unlink(missing_ok=True)

        df = df[df["Region"].isin(REGION_TO_SUFFIX)].copy()
        if "Dispatch Type" in df.columns:
            df = df[df["Dispatch Type"].astype(str).str.upper().isin(
                ["GENERATOR", "BIDIRECTIONAL", "BIDIRECTIONAL_UNIT", "GENERATING UNIT", "BIDIRECTIONAL UNIT"]
            )]
        df = df[df["DUID"].notna()].copy()
        df["DUID"] = df["DUID"].astype(str).str.strip()
        df = df[df["DUID"] != ""].drop_duplicates(subset=["DUID"], keep="last")

        duid_to_col = {}
        for _, row in df.iterrows():
            fuel_cols = ["Fuel Source - Primary", "Fuel Source - Descriptor", "Technology Type - Primary", "Technology Type - Descriptor"]
            text = " | ".join("" if pd.isna(v) else str(v).strip().lower() for v in (row.get(c) for c in fuel_cols))
            if   "battery" in text:                                          bucket = "battery"
            elif "wind" in text:                                             bucket = "wind"
            elif "solar" in text or "pv" in text:                           bucket = "solar"
            elif "hydro" in text:                                            bucket = "hydro"
            elif any(t in text for t in ["gas", "diesel", "distillate"]):   bucket = "gas"
            elif "coal" in text:                                             bucket = "coal"
            else: continue
            suffix = REGION_TO_SUFFIX.get(row["Region"])
            if suffix is None:
                continue
            duid_to_col[str(row["DUID"])] = f"battery_mw_{suffix}" if bucket == "battery" else f"{bucket}_mw_{suffix}"

        return duid_to_col

    def _API_call(duid_to_col):
        API_response = nemosis.dynamic_data_compiler(
            start_time=start,
            end_time=end,
            table_name="DISPATCH_UNIT_SCADA",
            raw_data_location="Pre_processing/temporary_cache",
            select_columns=["SETTLEMENTDATE", "DUID", "SCADAVALUE"],
            filter_cols=["DUID"],
            filter_values=[sorted(duid_to_col.keys())],
            fformat="feather",
            keep_csv=False,
        )
        return API_response

    def _clean_up(API_response, duid_to_col):
        battery_duids = {duid for duid, col in duid_to_col.items() if col.startswith("battery_mw_")}

        API_response["SETTLEMENTDATE"] = pd.to_datetime(API_response["SETTLEMENTDATE"])
        API_response["SCADAVALUE"]     = pd.to_numeric(API_response["SCADAVALUE"], errors="coerce")
        API_response["_col"]           = API_response["DUID"].map(duid_to_col)
        API_response = API_response[API_response["_col"].notna()].copy()

        API_response["_target"] = API_response["_col"]
        API_response["_value"]  = API_response["SCADAVALUE"].clip(lower=0)
        is_battery = API_response["DUID"].isin(battery_duids)
        if is_battery.any():
            charge    = is_battery & (API_response["SCADAVALUE"] < 0)
            discharge = is_battery & (API_response["SCADAVALUE"] >= 0)
            API_response.loc[charge,    "_target"] = API_response.loc[charge,    "_col"].str.replace("battery_mw_", "battery_charge_mw_",    regex=False)
            API_response.loc[charge,    "_value"]  = -API_response.loc[charge,   "SCADAVALUE"]
            API_response.loc[discharge, "_target"] = API_response.loc[discharge, "_col"].str.replace("battery_mw_", "battery_discharge_mw_", regex=False)
            API_response.loc[discharge, "_value"]  = API_response.loc[discharge, "SCADAVALUE"].clip(lower=0)

        API_response = (
            API_response.groupby(["SETTLEMENTDATE", "_target"])["_value"]
            .sum()
            .unstack("_target")
            .resample("5min").mean()
            .fillna(0)
        )

        # Ensure every expected fuel/region column is present (0 for months where
        # a fuel type had no registered/active DUIDs in DISPATCH_UNIT_SCADA).
        non_battery  = sorted({col for col in duid_to_col.values() if not col.startswith("battery_mw_")})
        battery_sfx  = sorted({col.replace("battery_mw_", "") for col in duid_to_col.values() if col.startswith("battery_mw_")})
        battery_cols = [f"battery_{d}_mw_{s}" for d in ("charge", "discharge") for s in battery_sfx]
        all_expected = sorted(set(non_battery + battery_cols))
        API_response = API_response.reindex(columns=all_expected, fill_value=0.0)

        return API_response

    duid_to_col = _build_mapping()
    data = _API_call(duid_to_col)
    data_clean = _clean_up(data, duid_to_col)
    return data_clean


In [269]:
"""
Datasource 4
 - Datasource origin: www.aemo.com.au
 - Datasource name: STTM_DWGM
    STTM   : AEMO's Short Term Trading Market — daily ex ante commodity price ($/GJ) at each gas hub, set by the market clearing process for the upcoming gas day
    DWGM   : AEMO's Declared Wholesale Gas Market — 5-interval-per-day scheduled price ($/GJ) for Victoria; each interval covers a ~4-hour scheduling horizon (6am, 10am, 2pm, 6pm, 10pm)
 - Variables:
    gas_price_nsw : STTM Sydney hub ex ante daily commodity price ($/GJ)
    gas_price_qld : STTM Brisbane hub ex ante daily commodity price ($/GJ)
    gas_price_sa  : STTM Adelaide hub ex ante daily commodity price ($/GJ)
    gas_price_vic : DWGM Victoria scheduled price ($/GJ) — 5 intervals per gas day, forward-filled within each ~4-hour scheduling horizon
"""

def _STTM_DWGM(start: str, end: str):
    

    def _load_1():
        request = urllib.request.Request(
            "https://www.aemo.com.au/-/media/files/gas/sttm/data/sttm-price-and-withdrawals.xlsx",
            headers={"User-Agent": "Mozilla/5.0"},
        )

        with urllib.request.urlopen(request, timeout=60) as response:
            data = openpyxl.load_workbook(io.BytesIO(response.read()), read_only=True, data_only=True)

        return data
    

    def _load_2():
        request = urllib.request.Request(
            "https://www.aemo.com.au/-/media/files/gas/dwgm/dwgm-prices-and-demand.xlsx",
            headers={"User-Agent": "Mozilla/5.0"},
        )

        with urllib.request.urlopen(request, timeout=60) as response:
            data = openpyxl.load_workbook(io.BytesIO(response.read()), read_only=True, data_only=True)

        return data
    

    def _process(data1, data2):

        idx = pd.date_range(start=start, end=end, freq="5min", name="SETTLEMENTDATE")

        sheets = {
            "SYD price and withdrawals": "gas_price_nsw",
            "ADL price and withdrawals": "gas_price_sa",
            "BRI price and withdrawals": "gas_price_qld",
        }

        data_clean = {}
        for sheet_name, col_name in sheets.items():
            rows = [(pd.Timestamp(r[0]), float(r[1])) for r in data1[sheet_name].iter_rows(min_row=2, values_only=True) if r[0] is not None and r[1] is not None]
            daily = pd.Series(dict(rows))
            data_clean[col_name] = daily.reindex(idx.normalize()).set_axis(idx).ffill().bfill()

        # DWGM (VIC) — 5 sub-daily intervals, ffill within each ~4-hour horizon
        rows_vic = [(pd.Timestamp(r[0]) + pd.Timedelta(hours=int(r[1])), float(r[2])) for r in data2["Prices"].iter_rows(min_row=2, values_only=True) if r[0] is not None and r[1] is not None and r[2] is not None]
        dwgm = pd.Series(dict(rows_vic), name="gas_price_vic").sort_index()
        data_clean["gas_price_vic"] = dwgm.reindex(idx, method="ffill").bfill()

        return pd.DataFrame(data_clean, index=idx)
    


    data1, data2 = _load_1(), _load_2()
    data_clean = _process(data1,data2)
    return data_clean


In [270]:
"""
Datasource 5
 - Datasource origin: www.visualcrossing.com
 - Datasource name: weather
 - Variables:
    temp             : Mean air temperature (°C)
    feelslike        : Apparent temperature combining heat index and wind chill (°C)
    dew              : Dew point temperature — measure of atmospheric moisture (°C)
    humidity         : Relative humidity (%)
    precip           : Total precipitation (rain/snow liquid equivalent) for the period (mm)
    precipprob       : Probability of precipitation (%)
    preciptype       : Type(s) of precipitation (rain, snow, freezing rain, ice)
    snow             : New snowfall amount (cm)
    snowdepth        : Depth of snow currently on the ground (cm)
    windgust         : Maximum short-term wind gust speed (kph)
    windspeed        : Mean wind speed (kph)
    winddir          : Wind direction — degrees clockwise from north (°)
    sealevelpressure : Atmospheric pressure normalised to sea level (mb)
    cloudcover       : Fraction of sky covered by cloud (%)
    visibility       : Horizontal visibility distance (km)
    solarradiation   : Instantaneous solar radiation at the surface (W/m²)
    solarenergy      : Total solar energy accumulated over the period (MJ/m²)
    uvindex          : UV exposure index (0–10)
    severerisk       : Probability of severe weather events (%)
    conditions       : Short text summary of weather conditions
    icon             : Weather icon identifier
    stations         : Weather station(s) used as source for the observation
"""

def _weather(start: str, end: str):
    
    def _load():
        sydney    = pd.read_csv("Pre_processing/weather_data/NSW weather.csv")
        brisbane  = pd.read_csv("Pre_processing/weather_data/QLD weather.csv")
        melbourne = pd.read_csv("Pre_processing/weather_data/VIC weather.csv")
        adelaide  = pd.read_csv("Pre_processing/weather_data/SA weather.csv")

        return sydney, brisbane, melbourne, adelaide

    def _process(data: pd.DataFrame, city: str) -> pd.DataFrame:
        data["datetime"] = pd.to_datetime(data["datetime"])
        data = data.set_index("datetime").sort_index()
        data = data[~data.index.duplicated(keep="first")]
        data = data.rename(columns={col: f"{str(col).strip().lower().replace(' ', '')}_{city}" for col in data.columns})
        data = data.apply(pd.to_numeric, errors="coerce")
        data = data.dropna(axis=1, how="all")
        return data.resample(f"{5}min").interpolate(method="time")
    
    sydney, brisbane, melbourne, adelaide = _load()

    sydney    = _process(sydney, "sydney")
    brisbane  = _process(brisbane, "brisbane")
    melbourne = _process(melbourne, "melbourne")
    adelaide  = _process(adelaide,  "adelaide")

   
    data_clean = pd.concat([sydney, brisbane, melbourne, adelaide], axis=1)
    data_clean.index.name = "SETTLEMENTDATE"
   
    return data_clean

   

In [271]:
def _nemseer_pull(
        start_date: pd.Timestamp,
        end_date: pd.Timestamp,
        datasource_file_path: Path,
        nemseer_forecast_type: str,
        nemseer_table_name: str,
        value_cols: list,
        run_col: str,
        interval_col: str,
        entity_col: str,
        ):

    def _repeat_logic(start: str, end: str):

        modified_start = (pd.Timestamp(start) - pd.Timedelta(days=1)).strftime("%Y/%m/%d %H:%M")

        def _API_call():
            # nemseer call requires two separate temp folders
            Path("Pre_processing/temporary_cache/raw").mkdir(parents=True, exist_ok=True)
            Path("Pre_processing/temporary_cache/processed").mkdir(parents=True, exist_ok=True)

            def get_special_dates():
                # Compute max allowed forecasted_end: nemseer caps at next-trading-day 04:00, Trading days run 04:00–04:00 AEST
                run_end_ts = pd.Timestamp(end[:16])
                current_trading_day_end = run_end_ts.normalize() + pd.Timedelta(hours=4)
                if run_end_ts >= current_trading_day_end:
                    current_trading_day_end += pd.Timedelta(days=1)
                return current_trading_day_end.strftime("%Y/%m/%d %H:%M")

            forecasted_end = get_special_dates()

            subprocess_code = textwrap.dedent(f"""
                import logging
                logging.getLogger("nemseer").setLevel(logging.WARNING)
                import nemseer, pandas as pd, sys
                data = nemseer.compile_data(
                    run_start={modified_start!r},
                    run_end={end[:16]!r},
                    forecasted_start={modified_start!r},
                    forecasted_end={forecasted_end!r},
                    raw_cache="Pre_processing/temporary_cache/raw",
                    processed_cache="Pre_processing/temporary_cache/processed",
                    forecast_type={nemseer_forecast_type!r},
                    tables=[{nemseer_table_name!r}],
                )
                df = data[{nemseer_table_name!r}]
                if df is None or df.empty:
                    raise ValueError("nemseer returned no data")
                df.reset_index().to_csv(sys.stdout, index=False)
            """)

            result = subprocess.run(
                [r"C:\Users\danie\conda_envs\nemseer_env\python.exe", "-c", subprocess_code],
                capture_output=True, text=True, cwd=Path.cwd(),
            )
            # Needs this to actually display errors from the subprocess
            if result.returncode != 0 or not result.stdout.strip():
                raise RuntimeError(f"nemseer subprocess failed:\n{result.stderr}")

            API_response = pd.read_csv(io.StringIO(result.stdout))
            return API_response


        def _clean_up(API_response):

            # Handle conversion of data types
            API_response[run_col]      = pd.to_datetime(API_response[run_col], format="mixed")
            API_response[interval_col] = pd.to_datetime(API_response[interval_col], format="mixed")
            for col in value_cols:
                API_response[col] = pd.to_numeric(API_response[col], errors="coerce")

            # Build 5-min output grid indexed by delivery timestamp
            output_idx = pd.date_range(
                start[:16], end[:16],
                freq="5min", name="SETTLEMENTDATE"
            )
            # Extend T_df back to `start` (1 day before output window) so that runs
            # are included in the pivot — their h57+ values can then
            # be ffilled forward into the Jan 1 rows
            full_idx = pd.date_range(
                pd.Timestamp(modified_start[:16]), pd.Timestamp(end[:16]),
                freq="5min", name="SETTLEMENTDATE"
            )
            T_df = pd.DataFrame({"SETTLEMENTDATE": full_idx})

            # For each delivery timestamp T, find the most recently published run (run_time <= T)
            # This is the causally correct assignment: no future run information leaks into T
            run_times_df = (
                API_response[[run_col]]
                .drop_duplicates()
                .sort_values(run_col)
                .reset_index(drop=True)
            )
            T_df = pd.merge_asof(
                T_df, run_times_df,
                left_on="SETTLEMENTDATE", right_on=run_col,
                direction="backward"
            )

            # Join each T to all forecast rows from its assigned run
            merged = T_df.merge(API_response, on=run_col, how="left")

            # Drop T values where no prior run exists (no run available before T)
            merged = merged[merged[interval_col].notna()]

            # Compute horizon relative to delivery timestamp T using ceiling division:
            # h=1 → first 30-min period strictly after T, h=2 → second, etc.
            # ceil is required (not round) so that e.g. T=00:15 with interval=00:30
            # gives (900s / 1800) = 0.5 → ceil → 1, not round → 0 (which would drop it)
            merged["horizon"] = (
                (merged[interval_col] - merged["SETTLEMENTDATE"]).dt.total_seconds()
                .div(1800)
                .apply(math.ceil)
            )

            # Keep only forward-looking horizons
            merged = merged[merged["horizon"] >= 1]

            # Pivot each value column separately and concat
            frames = []
            for col in value_cols:
                pivot = (
                    merged.groupby(["SETTLEMENTDATE", entity_col, "horizon"])[col]
                    .mean()
                    .unstack([entity_col, "horizon"])
                )
                pivot = pivot.sort_index(axis=1)

                if entity_col == "REGIONID":
                    # Strip trailing digit: "NSW1" → "nsw"
                    pivot.columns = [
                        f"{nemseer_forecast_type.lower()}_{col.lower()}_{ent[:-1].lower()}_h{h}"
                        for ent, h in pivot.columns
                    ]
                else:
                    # Interconnector: replace dashes with underscores, e.g. "NSW1-QLD1" → "nsw1_qld1"
                    pivot.columns = [
                        f"{nemseer_forecast_type.lower()}_{col.lower()}_{ent.lower().replace('-', '_')}_h{h}"
                        for ent, h in pivot.columns
                    ]
                frames.append(pivot)

            result = pd.concat(frames, axis=1)
            result.index.name = "SETTLEMENTDATE"

            # Reindex to the full 5-min grid, then ffill per horizon column:
            # carries the last known forecast for each horizon forward in time — no leakage
            # since ffill only looks backward (earlier timestamps)
            # full_idx covers the 3-day lookback so Dec 31 h57+ values ffill into Jan 1
            result = result.reindex(full_idx).ffill().reindex(output_idx)

            return result


        data = _API_call()
        data_clean = _clean_up(data)
        return data_clean

    _month_compiler(
        datasource_fetch_function = _repeat_logic,
        start_date = start_date,
        end_date = end_date,
        datasource_file_path = datasource_file_path
    )


In [272]:
"""
Datasource 8
 - Datasource origin: nemosis + www.aemo.com.au
 - Datasource name: bid stack   : Per-region aggregated energy bid offer stack from AEMO's BIDPEROFFER_D and
                                   BIDDAYOFFER_D tables, with DUID-to-region mapping from the NEM Registration List.
 - Variables:
    bidstack_mw_neg_{region}   : MW offered at negative prices (must-run / overflow)
    bidstack_mw_low_{region}   : MW offered $0–$300 (baseload)
    bidstack_mw_mid_{region}   : MW offered $300–$1 000
    bidstack_mw_high_{region}  : MW offered $1 000–$5 000
    bidstack_mw_vhigh_{region} : MW offered $5 000–$14 500
    bidstack_mw_cap_{region}   : MW offered ≥$14 500 (VoLL / price cap)
    bidstack_mw_total_{region} : Total MAXAVAIL across all DUIDs
"""
def _bidstack(start: str, end: str) -> pd.DataFrame:
    REGION_TO_SUFFIX = {"NSW1": "nsw", "QLD1": "qld", "VIC1": "vic", "SA1": "sa"}
    BAND_COLS  = [f"BANDAVAIL{i}" for i in range(1, 11)]
    PRICE_COLS = [f"PRICEBAND{i}" for i in range(1, 11)]
    PRICE_TIERS = [
        ("neg",    -float("inf"),    0.0),
        ("low",        0.0,        300.0),
        ("mid",      300.0,       1000.0),
        ("high",    1000.0,       5000.0),
        ("vhigh",  5000.0,      14500.0),
        ("cap",   14500.0,  float("inf")),
    ]

    def _build_duid_region():
        cache_path = Path("Pre_processing/temporary_cache") / "NEM Registration and Exemption List.xlsx"
        url = "https://www.aemo.com.au/-/media/files/electricity/nem/participant_information/nem-registration-and-exemption-list.xlsx"
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp, open(cache_path, "wb") as f:
            f.write(resp.read())
        try:
            df = pd.read_excel(cache_path, sheet_name="PU and Scheduled Loads", engine="openpyxl", header=0)
        finally:
            cache_path.unlink(missing_ok=True)
        df = df[df["Region"].isin(REGION_TO_SUFFIX) & df["DUID"].notna()].copy()
        df["DUID"] = df["DUID"].astype(str).str.strip()
        df = df[df["DUID"] != ""].drop_duplicates(subset=["DUID"], keep="last")
        return df.set_index("DUID")["Region"].map(REGION_TO_SUFFIX).to_dict()

    def _API_call():
        per = nemosis.dynamic_data_compiler(
            start_time=start, end_time=end,
            table_name="BIDPEROFFER_D",
            raw_data_location="Pre_processing/temporary_cache",
            select_columns=["INTERVAL_DATETIME", "DUID", "BIDTYPE"] + BAND_COLS + ["MAXAVAIL"],
            filter_cols=["BIDTYPE"], filter_values=[["ENERGY"]],
            fformat="feather", keep_csv=False, parse_data_types=False,
        )
        day = nemosis.dynamic_data_compiler(
            start_time=start, end_time=end,
            table_name="BIDDAYOFFER_D",
            raw_data_location="Pre_processing/temporary_cache",
            select_columns=["SETTLEMENTDATE", "DUID", "BIDTYPE"] + PRICE_COLS,
            filter_cols=["BIDTYPE"], filter_values=[["ENERGY"]],
            fformat="feather", keep_csv=False, parse_data_types=False,
        )
        return per, day

    def _clean_up(per, day, duid_to_region):
        per.columns = [c.upper() for c in per.columns]
        day.columns = [c.upper() for c in day.columns]

        per = per.rename(columns={"INTERVAL_DATETIME": "SETTLEMENTDATE"})
        per["SETTLEMENTDATE"] = pd.to_datetime(per["SETTLEMENTDATE"])
        day["SETTLEMENTDATE"] = pd.to_datetime(day["SETTLEMENTDATE"]).dt.normalize()

        for c in BAND_COLS + ["MAXAVAIL"]:
            per[c] = pd.to_numeric(per[c], errors="coerce").fillna(0.0).clip(lower=0.0)
        for c in PRICE_COLS:
            day[c] = pd.to_numeric(day[c], errors="coerce")

        d = per["SETTLEMENTDATE"].dt.normalize()
        per["TRADE_DAY"] = d.where(per["SETTLEMENTDATE"].dt.hour >= 4, d - pd.Timedelta(days=1))
        day = day.rename(columns={"SETTLEMENTDATE": "TRADE_DAY"})

        merged = per.merge(day[["DUID", "TRADE_DAY"] + PRICE_COLS], on=["DUID", "TRADE_DAY"], how="left")
        merged["region"] = merged["DUID"].map(duid_to_region)
        merged = merged.dropna(subset=["region"])

        for tier, lo, hi in PRICE_TIERS:
            merged[f"mw_{tier}"] = sum(
                merged[f"BANDAVAIL{i}"].where(
                    (merged[f"PRICEBAND{i}"] > lo) & (merged[f"PRICEBAND{i}"] <= hi), 0.0
                ) for i in range(1, 11)
            )

        tier_cols = [f"mw_{t}" for t, _, _ in PRICE_TIERS]
        agg = merged.groupby(["SETTLEMENTDATE", "region"])[tier_cols + ["MAXAVAIL"]].sum()
        agg = agg.rename(columns={**{f"mw_{t}": f"bidstack_mw_{t}" for t, _, _ in PRICE_TIERS},
                                    "MAXAVAIL": "bidstack_mw_total"})

        result = agg.unstack("region")
        result.columns = [f"{col}_{region}" for col, region in result.columns]
        result = result.resample("5min").mean().fillna(0)
        return result

    duid_to_region = _build_duid_region()
    per, day = _API_call()
    return _clean_up(per, day, duid_to_region)


In [273]:
"""
Datasource 1
"""
if False:
    _month_compiler(
        datasource_fetch_function = _dispatch_price,
        start_date = pd.Timestamp("2018/01/01"),
        end_date = pd.Timestamp("2026/01/01"),
        datasource_file_path = Path("Processed_data/1_dispatch_price.csv")
    )

In [274]:
"""
Datasource 2
"""
if False:
    _month_compiler(
        datasource_fetch_function = _dispatch_region_sum,
        start_date = pd.Timestamp("2018/01/01"),
        end_date = pd.Timestamp("2026/01/01"),
        datasource_file_path = Path("Processed_data/2_dispatch_region_sum.csv")
    )

In [275]:
"""
Datasource 3
"""
if False:
    _month_compiler(
        datasource_fetch_function = _generation_fuel,
        start_date = pd.Timestamp("2018/01/01"),
        end_date = pd.Timestamp("2018/02/01"),
        datasource_file_path = Path("Processed_data/3_generation_fuel.csv")
    )


In [276]:
"""
Datasource 4
"""
if False:
    _one_shot_compiler(
        datasource_fetch_function = _STTM_DWGM,
        start_date = pd.Timestamp("2018/01/01"),
        end_date = pd.Timestamp("2026/01/01"),
        datasource_file_path = Path("Processed_data/4_STTM_DWGM.csv")
    )

In [277]:
"""
Datasource 5
"""
if False:
    _one_shot_compiler(
        datasource_fetch_function = _weather,
        start_date = pd.Timestamp("2018/01/01"),
        end_date = pd.Timestamp("2026/01/01"),
        datasource_file_path = Path("Processed_data/5_weather.csv")
    )

In [278]:
"""
Datasource 6.1
 - Datasource origin: nemseer
 - Datasource name: predispatch price  : AEMO's PREDISPATCH PRICE table — contains 30-minute pre-dispatch pricing forecasts for each NEM region, published every half hour
 - Variables:
    RRP : Regional Reference Price forecast ($/MWh)
"""

if False:
    _nemseer_pull(
        start_date = pd.Timestamp("2018/01/01"),
        end_date = pd.Timestamp("2018/02/01"),
        datasource_file_path = Path("Processed_data/6_1_predispatch_price.csv"),
        nemseer_forecast_type = "PREDISPATCH",
        nemseer_table_name = "PRICE",
        value_cols = ["RRP"],
        run_col = "PREDISPATCH_RUN_DATETIME",
        interval_col = "DATETIME",
        entity_col = "REGIONID",
    )


In [279]:
"""
Datasource 6.2
 - Datasource origin: nemseer
 - Datasource name: predispatch region sum  : AEMO's PREDISPATCH REGIONSUM table — contains 30-minute pre-dispatch demand and generation forecasts for each NEM region, published every half hour
 - Variables:
    TOTALDEMAND         : Forecast total regional demand (MW)
    AVAILABLEGENERATION : Forecast available generation capacity (MW)
    NETINTERCHANGE      : Forecast net interconnector flow (MW)
    DEMANDFORECAST      : AEMO's internal demand forecast (MW)
    AVAILABLELOAD       : Forecast available scheduled load (MW)
"""

if True:
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2018/02/01"),
            datasource_file_path = Path("Processed_data/6_2_1_predispatch_regionsum.csv"),
            nemseer_forecast_type = "PREDISPATCH",
            nemseer_table_name = "REGIONSUM",
            value_cols = ["TOTALDEMAND"],
            run_col = "PREDISPATCH_RUN_DATETIME",
            interval_col = "DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2018/02/01"),
            datasource_file_path = Path("Processed_data/6_2_2_predispatch_regionsum.csv"),
            nemseer_forecast_type = "PREDISPATCH",
            nemseer_table_name = "REGIONSUM",
            value_cols = ["AVAILABLEGENERATION"],
            run_col = "PREDISPATCH_RUN_DATETIME",
            interval_col = "DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2018/02/01"),
            datasource_file_path = Path("Processed_data/6_2_3_predispatch_regionsum.csv"),
            nemseer_forecast_type = "PREDISPATCH",
            nemseer_table_name = "REGIONSUM",
            value_cols = ["NETINTERCHANGE"],
            run_col = "PREDISPATCH_RUN_DATETIME",
            interval_col = "DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2018/02/01"),
            datasource_file_path = Path("Processed_data/6_2_4_predispatch_regionsum.csv"),
            nemseer_forecast_type = "PREDISPATCH",
            nemseer_table_name = "REGIONSUM",
            value_cols = ["DEMANDFORECAST"],
            run_col = "PREDISPATCH_RUN_DATETIME",
            interval_col = "DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2018/02/01"),
            datasource_file_path = Path("Processed_data/6_2_5_predispatch_regionsum.csv"),
            nemseer_forecast_type = "PREDISPATCH",
            nemseer_table_name = "REGIONSUM",
            value_cols = ["AVAILABLELOAD"],
            run_col = "PREDISPATCH_RUN_DATETIME",
            interval_col = "DATETIME",
            entity_col = "REGIONID",
        )

In [280]:

"""
Datasource 6.3
 - Datasource origin: nemseer
 - Datasource name: predispatch interconnector solution  : AEMO's PREDISPATCH INTERCONNECTORRES table — contains 30-minute pre-dispatch MW flow forecasts for each NEM interconnector, published every half hour
 - Variables:
    MWFLOW        : Forecast MW flow on each interconnector
    METEREDMWFLOW : Metered MW flow forecast
"""

if True:
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2018/02/01"),
            datasource_file_path = Path("Processed_data/6_3_1_predispatch_interconnectorsoln.csv"),
            nemseer_forecast_type = "PREDISPATCH",
            nemseer_table_name = "INTERCONNECTORRES",
            value_cols = ["MWFLOW"],
            run_col = "PREDISPATCH_RUN_DATETIME",
            interval_col = "DATETIME",
            entity_col = "INTERCONNECTORID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2018/02/01"),
            datasource_file_path = Path("Processed_data/6_3_2_predispatch_interconnectorsoln.csv"),
            nemseer_forecast_type = "PREDISPATCH",
            nemseer_table_name = "INTERCONNECTORRES",
            value_cols = ["METEREDMWFLOW"],
            run_col = "PREDISPATCH_RUN_DATETIME",
            interval_col = "DATETIME",
            entity_col = "INTERCONNECTORID",
        )


In [281]:
"""
Datasource 7.1
 - Datasource origin: nemseer
 - Datasource name: pdpasa region solution  : AEMO's PDPASA REGIONSOLUTION table — contains 30-minute demand and reserve forecasts for each NEM region, published every half hour
 - Variables:
    DEMAND10                  : 10th percentile demand forecast (MW)
    DEMAND50                  : 50th percentile demand forecast (MW)
    DEMAND90                  : 90th percentile demand forecast (MW)
    RESERVEREQ                : Reserve requirement (MW)
    SURPLUSRESERVE            : Surplus reserve above requirement (MW)
    MAXSURPLUSRESERVE         : Maximum surplus reserve (MW)
    MAXSPARECAPACITY          : Maximum spare capacity headroom (MW)
    AGGREGATEPASAAVAILABILITY : Sum of PASA availability across all scheduled generators + semi-scheduled UIGF (MW)
"""

if True:
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2018/02/01"),
            datasource_file_path = Path("Processed_data/7_1_1_pdpasa_regionsolution.csv"),
            nemseer_forecast_type = "PDPASA",
            nemseer_table_name = "REGIONSOLUTION",
            value_cols = ["DEMAND10"],
            run_col = "RUN_DATETIME",
            interval_col = "INTERVAL_DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2018/02/01"),
            datasource_file_path = Path("Processed_data/7_1_2_pdpasa_regionsolution.csv"),
            nemseer_forecast_type = "PDPASA",
            nemseer_table_name = "REGIONSOLUTION",
            value_cols = ["DEMAND50"],
            run_col = "RUN_DATETIME",
            interval_col = "INTERVAL_DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2018/02/01"),
            datasource_file_path = Path("Processed_data/7_1_3_pdpasa_regionsolution.csv"),
            nemseer_forecast_type = "PDPASA",
            nemseer_table_name = "REGIONSOLUTION",
            value_cols = ["DEMAND90"],
            run_col = "RUN_DATETIME",
            interval_col = "INTERVAL_DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2018/02/01"),
            datasource_file_path = Path("Processed_data/7_1_4_pdpasa_regionsolution.csv"),
            nemseer_forecast_type = "PDPASA",
            nemseer_table_name = "REGIONSOLUTION",
            value_cols = ["RESERVEREQ"],
            run_col = "RUN_DATETIME",
            interval_col = "INTERVAL_DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2018/02/01"),
            datasource_file_path = Path("Processed_data/7_1_5_pdpasa_regionsolution.csv"),
            nemseer_forecast_type = "PDPASA",
            nemseer_table_name = "REGIONSOLUTION",
            value_cols = ["SURPLUSRESERVE"],
            run_col = "RUN_DATETIME",
            interval_col = "INTERVAL_DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2018/02/01"),
            datasource_file_path = Path("Processed_data/7_1_6_pdpasa_regionsolution.csv"),
            nemseer_forecast_type = "PDPASA",
            nemseer_table_name = "REGIONSOLUTION",
            value_cols = ["MAXSURPLUSRESERVE"],
            run_col = "RUN_DATETIME",
            interval_col = "INTERVAL_DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2018/02/01"),
            datasource_file_path = Path("Processed_data/7_1_7_pdpasa_regionsolution.csv"),
            nemseer_forecast_type = "PDPASA",
            nemseer_table_name = "REGIONSOLUTION",
            value_cols = ["MAXSPARECAPACITY"],
            run_col = "RUN_DATETIME",
            interval_col = "INTERVAL_DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2018/02/01"),
            datasource_file_path = Path("Processed_data/7_1_8_pdpasa_regionsolution.csv"),
            nemseer_forecast_type = "PDPASA",
            nemseer_table_name = "REGIONSOLUTION",
            value_cols = ["AGGREGATEPASAAVAILABILITY"],
            run_col = "RUN_DATETIME",
            interval_col = "INTERVAL_DATETIME",
            entity_col = "REGIONID",
        )


In [282]:
"""
Datasource 8
"""
if False:
    _month_compiler(
        datasource_fetch_function = _bidstack,
        start_date = pd.Timestamp("2018/01/01"),
        end_date = pd.Timestamp("2026/01/01"),
        datasource_file_path = Path("Processed_data/8_bidstack.csv")
    )


In [ ]:

def API_call():
    cache_path = Path("Pre_processing/temporary_cache") / "NEM Registration and Exemption List.xlsx"
    url = "https://www.aemo.com.au/-/media/files/electricity/nem/participant_information/nem-registration-and-exemption-list.xlsx"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(cache_path, "wb") as f:
        f.write(resp.read())
    try:
        df = pd.read_excel(cache_path, sheet_name="PU and Scheduled Loads", engine="openpyxl", header=0)
    finally:
        cache_path.unlink(missing_ok=True)
    return df

def classify_fuel(value):
    v = str(value).strip().lower()
    if "battery" in v:   return "battery"
    if "wind" in v:      return "wind"
    if "solar" in v:     return "solar"
    if "hydro" in v:     return "hydro"
    if any(t in v for t in ["gas", "diesel", "distillate"]): return "gas"
    if "coal" in v:      return "coal"
    return None

def clean_up(df):
    df = df[df["Dispatch Type"].astype(str).str.upper() != "LOAD"]
    df = df[df["DUID"].astype(str).str.upper() != "-"].copy()
    df["DUID"] = df["DUID"].astype(str).str.strip()
    df["Type"] = df["Fuel Source - Primary"].map(classify_fuel)
    return df

df = API_call()
df = clean_up(df)

df.to_excel("test.xlsx")
df


,Participant,Station Name,Region,Dispatch Type,Category,Classification,Fuel Source - Primary,Fuel Source - Descriptor,Technology Type - Primary,Technology Type - Descriptor,...,DUID,Reg Cap generation (MW),Max Cap generation (MW),Max ROC/Min generation,Reg Cap consumption (MW),Max Cap consumption (MW),Max ROC/Min consumption,Maximum storage capacity,Comments,Type
0,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Bidirectional Unit,Market,Scheduled,Battery Storage,Grid,Storage,Battery and Inverter,...,ADPBA1,7.76,6.15,4,7.0,6.0,4.0,6.0,NaN,battery
1,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Generating Unit,Market,Non-Scheduled,Hydro,Water,Renewable,Run of River,...,ADPMH1,1.44,1,-,NaN,NaN,NaN,NaN,NaN,hydro
2,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Generating Unit,Market,Semi-Scheduled,Solar,Solar,Renewable,Photovoltaic Tracking Flat panel,...,ADPPV1,24.75,19,4,NaN,NaN,NaN,NaN,NaN,solar
3,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Generating Unit,Market,Non-Scheduled,Solar,Solar,Renewable,Photovoltaic Flat panel,...,ADPPV2,0.2,0.2,-,NaN,NaN,NaN,NaN,NaN,solar
4,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Generating Unit,Market,Non-Scheduled,Solar,Solar,Renewable,Photovoltaic Flat panel,...,ADPPV3,0.02,0.02,-,NaN,NaN,NaN,NaN,NaN,solar
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
568,AGL Hydro Partnership,Yarrawonga Hydro Power Station,VIC1,Generating Unit,Market,Non-Scheduled,Hydro,Water,Renewable,Hydro - Gravity,...,YWNGAHYD,9,9,0,NaN,NaN,NaN,NaN,NaN,hydro
569,RTA Yarwun Pty Ltd,Yarwun Power Station,QLD1,Generating Unit,Market,Non-Scheduled**,Fossil,Natural Gas,Combustion,Combined Cycle Gas Turbine (CCGT),...,YARWUN_1,154,180,36,NaN,NaN,NaN,NaN,NaN,None
570,Yatpool Sun Farm Pty Ltd,Yatpool Solar Farm,VIC1,Generating Unit,Market,Semi-Scheduled,Solar,Solar,Renewable,Photovoltaic Tracking Flat panel,...,YATSF1,94,81,16,NaN,NaN,NaN,NaN,NaN,solar
571,Alinta Energy Retail Sales Pty Ltd,Yawong Wind Farm,VIC1,Generating Unit,Market,Non-Scheduled,Wind,Wind,Renewable,Wind - Onshore,...,YAWWF1,7.2,7.2,0,NaN,NaN,NaN,NaN,NaN,wind
